# E12 Digit Recognizer：高维 EDA、PCA 与错分分析

> Issue #9 / Epic #6 / 路线图 E12

## 你将学到
1. 高维像素的 EDA：看图，而不是看 64/784 列直方图
2. PCA 二维投影与方差解释
3. 混淆矩阵定位易混数字对，并抽查错误样本

## 数据
本地用 `sklearn.datasets.load_digits`（8×8）；Kaggle 28×28 同思路。


In [ ]:
QUICK = True
RANDOM_STATE = 42

# 一键就绪：关警告 + 打印环境 + 设种子（勿再叠 JE.no_warning / filterwarnings）
from wanglaoshi import ready
ready(seed=RANDOM_STATE)


## 1. 加载 digits


In [ ]:
from sklearn.datasets import load_digits
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

digits = load_digits()
X = digits.data
y = digits.target
print(X.shape, np.bincount(y))


## 2. 样本可视化


In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for ax, img, lab in zip(axes.ravel(), X[:16].reshape(-1, 8, 8), y[:16]):
    ax.imshow(img, cmap='gray')
    ax.set_title(str(lab))
    ax.axis('off')
plt.suptitle('Sample digits')
plt.tight_layout()
plt.show()


## 3. PCA 投影


In [ ]:
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
pca2 = PCA(n_components=2, random_state=RANDOM_STATE)
Z = pca2.fit_transform(Xtr)
plt.figure(figsize=(6, 5))
sc = plt.scatter(Z[:, 0], Z[:, 1], c=ytr, cmap='tab10', s=10)
plt.colorbar(sc)
plt.title(f'PCA 2D (var={pca2.explained_variance_ratio_.sum():.2%})')
plt.show()

pca = PCA(n_components=0.95, random_state=RANDOM_STATE).fit(Xtr)
print('components for 95% variance:', pca.n_components_)


## 4. QuickBaseline


In [ ]:
from wanglaoshi import CompetitionTools as CT

Xtr_df = pd.DataFrame(Xtr)
Xte_df = pd.DataFrame(Xte)
baseline = CT.QuickBaseline()
results = baseline.run_all_models(
    Xtr_df, pd.Series(ytr), Xte_df, pd.Series(yte), task_type='classification'
)
comparison = baseline.compare_models(results, top_n=5)
comparison


## 5. 混淆矩阵与易混对


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

best = comparison.iloc[0]['Model']
pred = np.asarray(results[best]['predictions'])
print(classification_report(yte, pred))
cm = confusion_matrix(yte, pred)
print(cm)

# 找非对角最大混淆
cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)
i, j = np.unravel_index(cm_off.argmax(), cm_off.shape)
print(f'most confused pair: true {i} -> pred {j} (count={cm_off[i, j]})')


## 6. 错误样本抽查


In [ ]:
wrong = np.where(pred != yte)[0][:8]
fig, axes = plt.subplots(1, max(1, len(wrong)), figsize=(12, 2))
if len(wrong) == 0:
    print('无错误样本')
else:
    if len(wrong) == 1:
        axes = [axes]
    for ax, idx in zip(axes, wrong):
        ax.imshow(Xte[idx].reshape(8, 8), cmap='gray')
        ax.set_title(f't{yte[idx]} p{pred[idx]}')
        ax.axis('off')
    plt.show()


## 7. 模拟提交


In [ ]:
from pathlib import Path
from wanglaoshi import CompetitionTools as CT

out = Path('outputs/12_digit_recognizer'); out.mkdir(parents=True, exist_ok=True)
sample = pd.DataFrame({'ImageId': np.arange(1, len(pred) + 1), 'Label': yte})
sample.to_csv(out / 'sample_submission.csv', index=False)
gen = CT.SubmissionGenerator()
sub = gen.create_submission(
    pred, str(out / 'sample_submission.csv'), str(out / 'submission.csv'), prediction_type='label'
)
print(gen.validate_submission(str(out / 'submission.csv'), str(out / 'sample_submission.csv')))
sub.head()


## 8. 检查清单
- [x] 样本可视化
- [x] PCA 投影与方差解释
- [x] 混淆矩阵 + 错误抽查
- [x] Label 列提交校验
- [ ] 对易混对做专门增强/模型
